In [1]:
import os
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-23.jdk/Contents/Home"  # or wherever your JDK 17/21 lives
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [2]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder.appName("Local Spark Session")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "spark-warehouse")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .enableHiveSupport()
    .getOrCreate()
)
print(spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/02 20:50:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/02 20:50:08 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


http://127.0.0.1:4041


# Join Types
* Inner joins (keep rows with keys that exist in the left and right datasets)
* Outer joins (keep rows with keys in either the left or right datasets)
* Left outer joins (keep rows with keys in the left dataset)
* Right outer joins (keep rows with keys in the right dataset)
* Left semi joins (keep the rows in the left, and only the left, dataset where the key appears in the right dataset)
* Left anti joins (keep the rows in the left, and only the left, dataset where they do not appear in the right dataset)
* Natural joins (perform a join by implicitly matching the columns between the two datasets with the same names)
* Cross (or Cartesian) joins (match every row in the left dataset with every row in the right dataset)

In [4]:
person = spark.createDataFrame(
    [
        (0,"Bill Chambers", 0, [100]),
        (1,"Matei Zaharia", 1, [500, 250, 100]),
        (2,"Michael Armbrust", 1, [250, 100])
    ]).toDF("id","name","graduate_program","spark_status")

graduateProgram = spark.createDataFrame(
    [
        (0, "Masters","School of Information", "UC Berkeley"),
        (2,"Masters","EECS","UC Berkeley"),
        (1,"Ph.D.","EECS","UC Berkeley")
    ]).toDF("id","degree","department","school")

sparkStatus = spark.createDataFrame(
    [
        (500,"Vice President"),
        (250,"PMC Member"),
        (100,"Contributor")
    ]).toDF("id","status")

person.createOrReplaceTempView("person")
graduateProgram.createOrReplaceTempView("graduateProgram")
sparkStatus.createOrReplaceTempView("sparkStatus")

In [ ]:
# inner joins
# Inner joins evaluate the keys in both of the Dataframes or table and include only the rows that evaluate
# to true
joinExpression = person["graduate_program"] == graduateProgram["id"]
person.join(graduateProgram, joinExpression).show() # Inner joins is the default join, thats why we don't specify

+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
| id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|  1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|  2|Michael Armbrust|               1|     [250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+



In [ ]:
# outer joins
# Outer joins evaluate the keys in both of the DataFrames or tables and include (and joins together)
# the rows that evaluate to true or false. If there is no equivalent row in either the left or right DataFrame
#, Spark will insert null

# Basically, a full outer join will not lose rows, and fill with null
# when we can't match

person.join(graduateProgram, joinExpression, "outer").show()

+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
|   0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|   1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|   2|Michael Armbrust|               1|     [250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|NULL|            NULL|            NULL|           NULL|  2|Masters|                EECS|UC Berkeley|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+



## Joins on Complex types
Even though this might seem like a challenge, it's actually not. Any expression is a valid join expression, assuming that it returns a Boolean.

In [ ]:
# Example
import pyspark.sql.functions as F
(
    person
    .withColumnRenamed("id", "personId")
    .join(sparkStatus, F.expr("array_contains(spark_status, id)"))
    .show()
)

# Fiquei na pagina 158

+--------+----------------+----------------+---------------+---+--------------+
|personId|            name|graduate_program|   spark_status| id|        status|
+--------+----------------+----------------+---------------+---+--------------+
|       0|   Bill Chambers|               0|          [100]|100|   Contributor|
|       1|   Matei Zaharia|               1|[500, 250, 100]|500|Vice President|
|       1|   Matei Zaharia|               1|[500, 250, 100]|250|    PMC Member|
|       1|   Matei Zaharia|               1|[500, 250, 100]|100|   Contributor|
|       2|Michael Armbrust|               1|     [250, 100]|250|    PMC Member|
|       2|Michael Armbrust|               1|     [250, 100]|100|   Contributor|
+--------+----------------+----------------+---------------+---+--------------+



## Communication Strategies

Spark approaches cluster communication in two different ways during joins. It either incurs a shuffle join, which results in an all-to-all communication or a broadcast join.

### BIG Table to BIG Table
When we join to big tables together, we occur in a shuffle join, which means that all partitions need to communicate with each other.

### Small Table to Big Table
When performing joins between 2 tables, where one is smaller then the other, we can improve the Performance by doing a Broadcast join, which will copy the small table to all the workers and perform the join operation there, without the need to move the data from the partitions to the driver. Which is very expensive and create a very big shuffle.

The image below show us an example:
![alt text](image.png)

We can set SQL to also perform an Broadcast join

```sql
SELECT /*+ MAPJOIN(graduateProgram) */ * FROM person JOIN graduateProgram
ON person.graduate_program = graduateProgram.id
```


In this chapter, we discussed joins, probably one of the most common use cases. One thing we did not mention but is important to consider is if you partition your data correctly prior to a join, you can end up with much more efficient execution because even if a shuffle is planned, if data from two different DataFrames is already located on the same machine, Spark can avoid the shuffle.